
# Machine Learning aplicado, Notebook 7
## Calibración de probabilidades: cuando el modelo dice 30%, tiene que ser 30%

**Curso:** *Machine Learning aplicado* · **Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De dónde venimos

En los notebooks 5 y 6 dijimos varias veces la misma frase: "el modelo deja de estar calibrado". La dijimos al
balancear las clases (la probabilidad promedio en testing subía de 0,33 a 0,48 cuando la tasa real de impago
era 0,30), la dijimos al comparar la logística con el boosting en Taiwán, y la dijimos al explicar por qué el
umbral óptimo de la fórmula de costos no coincidía con el de la tabla. Nunca mostramos cómo se arregla. Este
notebook cierra ese pendiente.

La idea es simple. Un modelo que predice probabilidades hace dos cosas distintas: **ordena** a los clientes
(los más riesgosos arriba) y les pone un **número**. El AUC mide solo lo primero. La calibración mide lo
segundo: si tomo a todos los clientes a los que el modelo les dio 30%, ¿deja de pagar el 30% de ellos? Para
aprobar o rechazar con un umbral cualquiera basta ordenar bien; para fijar un umbral con costos, para
provisionar, para cobrar una tasa según el riesgo, o para sumar riesgos de una cartera, el número tiene que
significar lo que dice.

### Qué vas a aprender hoy

1. Cómo se mide la calibración: el diagrama de confiabilidad, el Brier score y su descomposición, el log loss y el error de calibración esperado (ECE).
2. Por qué se pierde: al balancear clases o ponderar filas, por la forma del modelo (una logística mal especificada), y en los árboles y ensambles por construcción.
3. Cómo se corrige cuando la causa es el balanceo: la corrección analítica del intercepto, exacta para la logística y sin datos extra.
4. Cómo se corrige en general: Platt scaling y regresión isotónica, siempre sobre datos que el modelo no vio, y qué hace `CalibratedClassifierCV`.
5. Qué cambia en la decisión: con probabilidades calibradas, el umbral de la fórmula de costos vuelve a ser confiable, y la probabilidad sirve para provisionar y para ponerle precio al riesgo.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)



## 1. Las dos bases, preparadas igual que antes

Usamos las mismas dos bases de los notebooks 5 y 6, con la misma preparación y la misma partición, para que los
números se puedan comparar celda por celda. **German Credit**: 1.000 créditos, 300 impagos, 18 columnas, 700
de training y 300 de testing. **Taiwán**: 30.000 tarjetas, 22% de impago, 23 columnas, 21.000 de training y
9.000 de testing. Las celdas de carga son las de siempre; si tienes los CSV del curso, pega sus links.


In [ ]:

URL_GERMAN = ""    # <-- pega aquí el link del CSV del curso (german_credit.csv) si lo tienes

def normalizar_link(url):
    '''Convierte un link de Dropbox o Google Drive en un link de descarga directa.'''
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_GERMAN:
    german = pd.read_csv(normalizar_link(URL_GERMAN))
else:
    german = fetch_openml("credit-g", version=1, as_frame=True).frame
    for columna in german.select_dtypes("category").columns:
        german[columna] = german[columna].astype(str)
german = german.reset_index(drop=True)
german.insert(0, "id", np.arange(1, len(german) + 1))
print("Clientes:", len(german), "  Columnas:", german.shape[1] - 1)
print("Proporción de malos pagadores:", round((german["class"] == "bad").mean(), 3))
german[["id", "duration", "credit_amount", "age", "checking_status", "credit_history", "savings_status", "class"]].head(8)


In [ ]:

T = pd.DataFrame({"id": german["id"], "y": (german["class"] == "bad").astype(int)})
T["duracion"] = german["duration"].astype(float)                     # meses
T["monto"] = german["credit_amount"].astype(float)                   # marcos alemanes (DM)
T["edad"] = german["age"].astype(float)
T["cuota_pct_ingreso"] = german["installment_commitment"].astype(float)   # tramo 1 a 4 de la cuota como % del ingreso

cs = german["checking_status"]                                       # base: no tiene cuenta corriente
T["cta_negativa"] = (cs == "<0").astype(int)
T["cta_0_200"] = (cs == "0<=X<200").astype(int)
T["cta_sobre_200"] = (cs == ">=200").astype(int)
ch = german["credit_history"]                                        # base: créditos vigentes al día
T["hist_critico"] = (ch == "critical/other existing credit").astype(int)
T["hist_atrasos"] = (ch == "delayed previously").astype(int)
T["hist_sin_creditos"] = ch.isin(["all paid", "no credits/all paid"]).astype(int)
sv = german["savings_status"]                                        # base: ahorro menor a 100 DM
T["ahorro_100_1000"] = sv.isin(["100<=X<500", "500<=X<1000"]).astype(int)
T["ahorro_sobre_1000"] = (sv == ">=1000").astype(int)
T["ahorro_desconocido"] = (sv == "no known savings").astype(int)
hs = german["housing"]                                               # base: vivienda propia
T["vivienda_arriendo"] = (hs == "rent").astype(int)
T["vivienda_gratis"] = (hs == "for free").astype(int)
pu = german["purpose"]                                               # base: otros propósitos
T["proposito_auto"] = pu.isin(["new car", "used car"]).astype(int)
T["proposito_radio_tv"] = (pu == "radio/tv").astype(int)
T["proposito_muebles"] = (pu == "furniture/equipment").astype(int)

XCOLS = [c for c in T.columns if c not in ("id", "y")]
NUM = ["duracion", "monto", "edad", "cuota_pct_ingreso"]
print(len(XCOLS), "columnas explicativas:", XCOLS)
T.head()


In [ ]:

rng = np.random.default_rng(2026)
conjunto = np.array(["training"] * len(T), dtype=object)
for clase in (0, 1):
    idx = np.where(T["y"].values == clase)[0]
    rng.shuffle(idx)
    conjunto[idx[: int(round(0.3 * len(idx)))]] = "testing"
T["conjunto"] = conjunto
tr = T["conjunto"] == "training"
te = ~tr

media = T.loc[tr, NUM].mean()
desv = T.loc[tr, NUM].std(ddof=0)      # desviación poblacional, la misma que usa StandardScaler
S = T.copy()
S[NUM] = (T[NUM] - media) / desv

X_tr, y_tr = S.loc[tr, XCOLS].values, S.loc[tr, "y"].values
X_te, y_te = S.loc[te, XCOLS].values, S.loc[te, "y"].values
print("training:", tr.sum(), "clientes,", y_tr.sum(), "malos pagadores  |  testing:", te.sum(), "clientes,", y_te.sum(), "malos pagadores")
pd.DataFrame({"promedio training": media.round(3), "desviación training": desv.round(3)})


In [ ]:

URL_TAIWAN = ""    # <-- pega aquí el link del CSV del curso (taiwan_credit.csv) si lo tienes

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_TAIWAN:
    tw = pd.read_csv(normalizar_link(URL_TAIWAN))
    X_tw = tw.drop(columns=tw.columns[-1]).apply(pd.to_numeric)
    y_tw = tw[tw.columns[-1]].astype(int).values
else:
    tw = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
    X_tw = tw.data.apply(pd.to_numeric)
    y_tw = tw.target.astype(int).values

meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X_tw.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
             + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
print("Clientes:", len(X_tw), "  Columnas:", X_tw.shape[1], "  Proporción que no pagó:", round(y_tw.mean(), 3))



In [ ]:

X_tw_tr, X_tw_te, yt_tr, yt_te = train_test_split(X_tw, y_tw, test_size=0.3, random_state=42, stratify=y_tw)
esc_tw = StandardScaler().fit(X_tw_tr)
A_tr, A_te = esc_tw.transform(X_tw_tr), esc_tw.transform(X_tw_te)
print("Taiwán, training:", len(yt_tr), "clientes,", int(yt_tr.sum()), "no pagaron  |  testing:", len(yt_te), "clientes,", int(yt_te.sum()), "no pagaron")



## 2. Qué es estar calibrado y cómo se mide

Un modelo está **calibrado** si, entre los clientes a los que les asigna una probabilidad $p$, la fracción que
efectivamente resulta positiva es $p$. Con probabilidades continuas no hay "todos los que tienen exactamente
0,30", así que se agrupan: se ordenan los clientes de testing por $p$, se parten en diez grupos del mismo
tamaño (deciles), y en cada grupo se compara la $p$ promedio con la fracción real de positivos. Dibujar esos
diez puntos contra la diagonal es el **diagrama de confiabilidad**: puntos sobre la diagonal, calibrado; por
debajo, el modelo exagera el riesgo; por encima, lo subestima.

Tres números resumen el diagrama:

- **Brier score** $= \frac{1}{n}\sum_i (p_i - y_i)^2$, el error cuadrático medio de la probabilidad contra el
  resultado 0/1. Se descompone en un término de calibración (qué tan lejos de la diagonal están los puntos)
  más un término de resolución (qué tan distintos son los deciles entre sí) más la varianza de $y$, que no
  depende del modelo. Bajar el Brier puede venir de calibrar mejor o de discriminar mejor.
- **Log loss**, la entropía cruzada promedio del notebook 5: castiga con fuerza la seguridad equivocada. Un
  modelo que dice 0,0001 y falla paga 9,2; uno que dice 0,01 paga 4,6.
- **ECE** (expected calibration error): el promedio, ponderado por el tamaño de cada decil, de la distancia
  vertical entre el punto y la diagonal. Es la forma más directa de poner un número al diagrama. Con diez
  deciles de 30 clientes (German Credit) tiene bastante ruido; con deciles de 900 (Taiwán) es confiable.

La regla que ninguno de los tres rompe: el AUC no cambia si le aplico a las probabilidades una transformación
creciente cualquiera (el orden es el mismo), pero el Brier, el log loss y el ECE sí. Por eso puede haber dos
modelos con el mismo AUC y calibraciones muy distintas, y por eso calibrar nunca mejora el orden.


In [ ]:

def confiabilidad(y_real, p, k=10):
    '''Tabla por deciles de p: cuántos clientes, p promedio, fracción real de positivos.'''
    q = pd.qcut(p, k, labels=False, duplicates="drop")
    return pd.DataFrame({"decil": q, "p": p, "y": y_real}).groupby("decil").agg(n=("y", "size"), p_promedio=("p", "mean"), fraccion_real=("y", "mean"))

def ece(y_real, p, k=10):
    t = confiabilidad(y_real, p, k)
    return float((t.n / len(y_real) * (t.fraccion_real - t.p_promedio).abs()).sum())

def metricas(y_real, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(AUC=roc_auc_score(y_real, p), Brier=brier_score_loss(y_real, p), log_loss=log_loss(y_real, p), ECE=ece(y_real, p), p_promedio=p.mean(), tasa_real=y_real.mean())

def diagrama(y_real, modelos, titulo):
    '''Diagrama de confiabilidad: un trazo por modelo, {nombre: p}.'''
    fig = go.Figure()
    fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="calibración perfecta", line=dict(dash="dash", color="gray"))
    for nombre, p in modelos.items():
        t = confiabilidad(y_real, p)
        fig.add_scatter(x=t.p_promedio, y=t.fraccion_real, mode="lines+markers", name=nombre)
    fig.update_layout(title=titulo, xaxis_title="probabilidad predicha (promedio del decil)", yaxis_title="fracción real de positivos",
                      xaxis_range=[0, 1], yaxis_range=[0, 1], height=460, width=620)
    return fig

def tabla_metricas(y_real, modelos):
    return pd.DataFrame({n: metricas(y_real, p) for n, p in modelos.items()}).T.round(4)

base = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr)
p_base = base.predict_proba(X_te)[:, 1]
print(confiabilidad(y_te, p_base).round(3))
print()
print(pd.Series(metricas(y_te, p_base)).round(4))
diagrama(y_te, {"logística sin balancear": p_base}, "German Credit, testing: diagrama de confiabilidad").show()



La logística del notebook 5 está razonablemente calibrada: la probabilidad promedio en testing (0,332) se
parece a la tasa real (0,300), los deciles bajos y altos caen cerca de la diagonal, y el ECE de 0,052 es
compatible con el ruido de deciles de 30 clientes. Es lo que se espera de una logística entrenada con la
entropía cruzada sin ponderar: la función de costo que minimiza es exactamente la que premia la calibración.
Eso va a cambiar en cuanto la ponderemos.



## 3. Por qué se pierde la calibración

### Causa 1: balancear las clases

En el notebook 5 corregimos el desbalanceo con `class_weight="balanced"` (peso 1,667 para los malos pagadores
y 0,714 para los buenos) y vimos que el recall subía. El precio, que dijimos y no mostramos, es que el modelo
aprende con una base ficticia donde los malos pagadores son el 50%, y devuelve probabilidades de esa base
ficticia. El orden de los clientes casi no cambia (el AUC es el mismo), pero cada número está inflado.


In [ ]:

cw = LogisticRegression(penalty=None, max_iter=5000, class_weight="balanced").fit(X_tr, y_tr)
p_cw = cw.predict_proba(X_te)[:, 1]
n1, n0 = int(y_tr.sum()), int((y_tr == 0).sum())
w1, w0 = len(y_tr) / (2 * n1), len(y_tr) / (2 * n0)
print(f"pesos 'balanced': w1 = {w1:.3f}, w0 = {w0:.3f}, razón w1/w0 = {w1/w0:.3f} (= n0/n1)")
print(tabla_metricas(y_te, {"sin balancear": p_base, "class weights": p_cw}))
diagrama(y_te, {"sin balancear": p_base, "class weights": p_cw}, "German Credit: el modelo balanceado infla todas las probabilidades").show()



El diagrama lo dice todo: la curva de class weights queda entera por debajo de la diagonal (el modelo dice
más riesgo del que hay), el ECE salta de 0,05 a 0,18 y el Brier de 0,151 a 0,188, con el mismo AUC. Lo
mismo pasa con el sobremuestreo (repetir filas es ponderarlas) y con el submuestreo. No es un defecto del
balanceo: es su consecuencia lógica, porque le cambiamos la tasa base al modelo.

### Causa 2: la forma del modelo

Una logística lineal supone que el logaritmo de las chances es lineal en las variables. Si la realidad tiene
escalones, como en Taiwán (el impago salta de 13% a 33% con un mes de atraso y a 68% con dos), la logística
dibuja una rampa que pasa por encima de la realidad en un tramo y por debajo en otro. Está mal especificada,
y un modelo mal especificado puede estar descalibrado aunque se haya entrenado con la entropía cruzada.

### Causa 3: los árboles y los ensambles, por construcción

Un árbol sin límite termina con una empresa por hoja y predice 0 o 1: probabilidades extremas que casi
siempre están mal, y el log loss lo castiga sin piedad. Un random forest promedia cientos de árboles que
memorizan, y ese promedio tiende a comprimir las probabilidades hacia el centro o a exagerarlas en los
extremos, según cómo crecieron los árboles. El boosting, que minimiza la entropía cruzada en cada ronda,
suele salir mejor calibrado, pero tampoco tiene garantía.

Para ver las tres cosas de una vez entrenamos en Taiwán la logística, un árbol sin límite, el árbol de
profundidad 4, dos random forest (uno con hojas de 1 cliente y otro con hojas de 5, el del notebook 6) y el
boosting. Un detalle que importa desde ya: para la sección 5 vamos a necesitar datos que el modelo no haya
visto y que no sean testing, así que training se parte en dos desde ahora: **ajuste** (70% de training) y
**calibración** (30%). Todos los modelos de este notebook se entrenan con la parte de ajuste.


In [ ]:

A_fit, A_cal, y_fit, y_cal = train_test_split(A_tr, yt_tr, test_size=0.3, random_state=1, stratify=yt_tr)
print("ajuste:", len(y_fit), " calibración:", len(y_cal), " testing:", len(yt_te))

modelos_tw = {
    "logística": LogisticRegression(penalty=None, max_iter=5000),
    "árbol sin límite": DecisionTreeClassifier(random_state=0),
    "árbol profundidad 4": DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=0),
    "random forest, hojas de 1": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0),
    "random forest, hojas de 5": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "gradient boosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0),
}
P_cal, P_te = {}, {}
t0 = time.time()
for nombre, m in modelos_tw.items():
    m.fit(A_fit, y_fit)
    P_cal[nombre] = m.predict_proba(A_cal)[:, 1]
    P_te[nombre] = m.predict_proba(A_te)[:, 1]
print(f"({time.time() - t0:.0f} s)")
print(tabla_metricas(yt_te, P_te))
diagrama(yt_te, {k: P_te[k] for k in ["logística", "random forest, hojas de 1", "gradient boosting"]}, "Taiwán, testing: tres modelos sin calibrar").show()
print(confiabilidad(yt_te, P_te["logística"]).round(3))



Tres lecturas. **La logística es la peor calibrada** de las que predicen algo (ECE 0,051): en los deciles del
medio dice 18% a 23% y la realidad es 12% a 15%, y en el decil más alto dice 59% y pasa 66%; es la rampa
contra la escalera. **El árbol sin límite** es un desastre de calibración aunque su ECE por deciles parezca
bajo (predice 0 o 1, así que los deciles no se pueden armar; mira el Brier de 0,27 y el log loss de 3,7,
contra 0,14 y 0,45 de los demás). **Los ensambles están mejor de lo que uno esperaría**: el boosting y el
random forest con hojas de 5 tienen ECE de 0,012 a 0,014, y el random forest con hojas de 1 exagera un poco
en las puntas (ECE 0,030). Con 14.700 filas de ajuste y hojas mínimas razonables, los ensambles de este
problema salen bastante calibrados solos. No siempre es así, y por eso hay que mirar el diagrama antes de
decidir si vale la pena corregir.



## 4. Corrección analítica del intercepto: deshacer el balanceo sin datos extra

Cuando la descalibración viene de haber ponderado las clases, la logística tiene una solución exacta que no
necesita ni un dato más. Ponderar la clase 1 por $w_1$ y la clase 0 por $w_0$ equivale a entrenar sobre una
población donde las chances a priori de ser positivo se multiplicaron por $w_1 / w_0$. En una logística eso
desplaza solo el intercepto: el logit que aprende el modelo balanceado es el logit verdadero más
$\ln(w_1 / w_0)$. Para volver a la población real basta restar esa constante:

$$z_{\text{corregido}} = z_{\text{balanceado}} - \ln\frac{w_1}{w_0}, \qquad p_{\text{corregido}} = \frac{1}{1 + e^{-z_{\text{corregido}}}}$$

Con `class_weight="balanced"` la razón $w_1 / w_0 = n_0 / n_1 = 490/210 = 2{,}333$ y la constante es
$\ln 2{,}333 = 0{,}847$. Con sobremuestreo de la minoritaria repetida $k$ veces la constante es $\ln k$; con
submuestreo que deja $n_0'$ buenos, $\ln(n_0 / n_0')$. El orden de los clientes no cambia (es restar una
constante al logit), así que el AUC queda igual y solo se corrige el nivel. Es el resultado de King y Zeng
(2001) para muestreo por clase, y en la práctica es lo primero que hay que hacer si balanceaste una logística.


In [ ]:

z_cw = cw.decision_function(X_te)                      # el logit del modelo balanceado
p_corr = 1 / (1 + np.exp(-(z_cw - np.log(w1 / w0))))   # restar ln(w1/w0) = ln(490/210)
print(f"constante restada: ln({w1/w0:.3f}) = {np.log(w1/w0):.3f}")
print(tabla_metricas(y_te, {"sin balancear": p_base, "class weights": p_cw, "class weights + corrección": p_corr}))
diagrama(y_te, {"sin balancear": p_base, "class weights": p_cw, "class weights + corrección": p_corr}, "German Credit: la corrección del intercepto devuelve la curva a la diagonal").show()

# lo mismo con sobremuestreo: cada malo pagador dos veces equivale a w1/w0 = 2, constante ln 2
over = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr, sample_weight=np.where(y_tr == 1, 2.0, 1.0))
z_over = over.decision_function(X_te)
print()
print(tabla_metricas(y_te, {"sobremuestreo x2": 1 / (1 + np.exp(-z_over)), "sobremuestreo x2 + corrección ln 2": 1 / (1 + np.exp(-(z_over - np.log(2))))}))



La curva corregida cae encima de la del modelo sin balancear: mismo AUC de 0,824, Brier de vuelta a 0,151 y
ECE 0,045, con probabilidad promedio 0,332. Balanceamos para mover el punto de operación y al final
recuperamos las probabilidades reales: las dos cosas se pueden tener. (Y es otra forma de ver la conclusión
del notebook 5: para una logística, balancear y luego corregir el intercepto es lo mismo que no balancear y
mover el umbral.)



## 5. Corrección general: Platt scaling y regresión isotónica

Cuando la causa no es un peso conocido (la logística mal especificada, los árboles, un modelo que viene de
otra parte), no hay constante que restar. La solución general es **aprender una función que traduzca los
scores del modelo a probabilidades reales**, usando datos etiquetados que el modelo no vio. Dos funciones son
las de siempre:

- **Platt scaling** (Platt, 1999): una logística de un solo insumo, el logit del modelo, $p = 1/(1 + e^{-(a z + b)})$.
  Dos parámetros, así que funciona con pocos datos, y solo puede estirar o comprimir y desplazar: arregla un
  modelo uniformemente demasiado seguro o demasiado tibio, pero no puede arreglar una rampa contra una
  escalera, porque la transformación es una sigmoide.
- **Regresión isotónica** (Zadrozny y Elkan, 2002): la función creciente en escalera que mejor ajusta la
  fracción real de positivos contra el score. No tiene forma fija, así que puede corregir cualquier
  deformación monótona, pero necesita miles de filas y puede sobreajustar con pocas.

La regla que no se negocia: la función de calibración se ajusta sobre la parte de **calibración** (o por
validación cruzada), nunca sobre las filas con que se ajustó el modelo (en esas el modelo se ve mejor de lo
que es) y nunca sobre testing (que tiene que quedar limpio para medir). Por eso partimos training en dos en
la sección 3. En scikit-learn, `CalibratedClassifierCV(modelo, method="sigmoid" | "isotonic", cv=5)` hace
esto por validación cruzada y además promedia los cinco modelos, así que sus resultados mezclan calibración
con un poco de ensamble; aquí lo hacemos a mano para que se vea cada pieza.


In [ ]:

def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))

def calibrar_platt(p_cal, y_cal, p_nuevo):
    f = LogisticRegression().fit(logit(p_cal).reshape(-1, 1), y_cal)      # a·z + b
    return f.predict_proba(logit(p_nuevo).reshape(-1, 1))[:, 1]

def calibrar_isotonica(p_cal, y_cal, p_nuevo):
    f = IsotonicRegression(out_of_bounds="clip").fit(p_cal, y_cal)
    return f.predict(p_nuevo)

filas = []
P_platt, P_iso = {}, {}
for nombre in modelos_tw:
    P_platt[nombre] = calibrar_platt(P_cal[nombre], y_cal, P_te[nombre])
    P_iso[nombre] = calibrar_isotonica(P_cal[nombre], y_cal, P_te[nombre])
    for etiqueta, p in [("sin calibrar", P_te[nombre]), ("Platt", P_platt[nombre]), ("isotónica", P_iso[nombre])]:
        m = metricas(yt_te, p); filas.append(dict(modelo=nombre, calibración=etiqueta, AUC=m["AUC"], Brier=m["Brier"], log_loss=m["log_loss"], ECE=m["ECE"]))
tabla_cal = pd.DataFrame(filas).set_index(["modelo", "calibración"]).round(4)
print(tabla_cal.to_string())


In [ ]:

diagrama(yt_te, {"logística": P_te["logística"], "logística + Platt": P_platt["logística"], "logística + isotónica": P_iso["logística"]},
         "Taiwán: la logística mal especificada, Platt no la arregla e isotónica sí").show()
diagrama(yt_te, {"random forest, hojas de 1": P_te["random forest, hojas de 1"], "+ Platt": P_platt["random forest, hojas de 1"], "+ isotónica": P_iso["random forest, hojas de 1"]},
         "Taiwán: el random forest de hojas de 1 exagera en las puntas, y las dos correcciones lo acercan").show()



Cómo leer la tabla y los dos diagramas:

- **La logística**: Platt no hace nada (ECE 0,053 contra 0,051), porque una sigmoide sobre una sigmoide sigue
  siendo una rampa. La isotónica sí: ECE de 0,051 a 0,009, Brier de 0,146 a 0,143, y el diagrama se pega a la
  diagonal. Es la corrección de forma que Platt no puede hacer. Fíjate en que el AUC casi no cambia (0,715 a
  0,717): calibrar no mejora el orden, y la pequeña diferencia viene de los empates que crea la escalera.
- **El árbol sin límite**: Brier de 0,27 a 0,16 y log loss de 3,7 a 0,50 con cualquiera de las dos, pero el
  AUC sigue en 0,616. La calibración arregla los números, no la ranking; un modelo que ordena mal sigue
  ordenando mal con probabilidades bonitas.
- **Random forest de hojas de 1**: ECE de 0,030 a 0,019 (Platt) y 0,013 (isotónica), con el Brier apenas mejor.
- **Los que ya estaban calibrados** (árbol de profundidad 4, random forest de hojas de 5, boosting): las
  correcciones cambian el Brier en la cuarta cifra y la isotónica a veces empeora el log loss. Calibrar un
  modelo que no lo necesita no es gratis: la función de calibración también tiene error muestral.

La regla práctica que sale de aquí: primero el diagrama; si la curva está lejos de la diagonal en forma de
desplazamiento uniforme, Platt (o la corrección del intercepto si la causa fue un peso); si está deformada y
hay miles de filas, isotónica; si está pegada a la diagonal, no tocar.



## 6. Qué cambia en la decisión: la fórmula de costos vuelve a funcionar

En el notebook 5 vimos que con un falso negativo cinco veces más caro que un falso positivo el umbral óptimo
de la **fórmula** es $c_{FP} / (c_{FP} + c_{FN}) = 1/6 = 0{,}167$, y que la **tabla** sobre testing daba otro
número. Dijimos que una de las razones era la calibración: la fórmula supone que $p$ es la probabilidad real
de impago, y si $p$ está inflada, el umbral de la fórmula corta donde no debe. Ahora lo podemos comprobar
con el modelo balanceado, antes y después de la corrección, y con el random forest de Taiwán, antes y después
de Platt.


In [ ]:

def costo_por_umbral(y_real, p, c_fn, c_fp, umbrales=np.round(np.arange(0.05, 0.96, 0.05), 2)):
    return pd.Series({u: int(((p > u) & (y_real == 0)).sum() * c_fp + ((p <= u) & (y_real == 1)).sum() * c_fn) for u in umbrales})

def comparar_umbrales(y_real, modelos, c_fn, c_fp):
    u_formula = c_fp / (c_fp + c_fn)
    filas = []
    for nombre, p in modelos.items():
        tabla = costo_por_umbral(y_real, p, c_fn, c_fp)
        costo_formula = int(((p > u_formula) & (y_real == 0)).sum() * c_fp + ((p <= u_formula) & (y_real == 1)).sum() * c_fn)
        filas.append(dict(modelo=nombre, umbral_formula=round(u_formula, 3), costo_en_formula=costo_formula, umbral_tabla=float(tabla.idxmin()), costo_minimo_tabla=int(tabla.min()),
                          p_promedio=round(p.mean(), 3)))
    return pd.DataFrame(filas).set_index("modelo")

print("German Credit, c_FN = 5, c_FP = 1:")
print(comparar_umbrales(y_te, {"sin balancear": p_base, "class weights": p_cw, "class weights + corrección": p_corr}, c_fn=5, c_fp=1))
print()
print("Taiwán, c_FN = 4, c_FP = 1:")
print(comparar_umbrales(yt_te, {"random forest, hojas de 1": P_te["random forest, hojas de 1"], "+ Platt": P_platt["random forest, hojas de 1"], "+ isotónica": P_iso["random forest, hojas de 1"],
                                "gradient boosting": P_te["gradient boosting"], "gradient boosting + Platt": P_platt["gradient boosting"]}, c_fn=4, c_fp=1))



Con el modelo balanceado sin corregir, la fórmula (0,167) es un desastre: corta muy abajo en una escala
inflada y el costo sube a 182, mientras la tabla encuentra su mínimo recién en 0,50. Corregido el intercepto,
la fórmula vuelve a dar un costo (149) cercano al mínimo de la tabla (133 en 0,25), igual que con el modelo
sin balancear. La diferencia que queda entre 0,167 y 0,25 es la otra razón que dimos en el notebook 5: la
variación muestral de una tabla armada con 300 clientes.

En Taiwán, con 9.000 clientes de testing, la tabla es mucho menos ruidosa y los modelos ya estaban casi
calibrados, así que la historia es distinta y también instructiva. Para el boosting, fórmula y tabla dicen lo
mismo (0,20) antes y después de Platt: no había nada que corregir. Para el random forest de hojas de 1 la
fórmula dice 0,20 y la tabla 0,25, y calibrar no mueve el mínimo de la tabla, porque la descalibración era
pequeña (ECE 0,03) y estaba en las puntas, no alrededor de 0,2. Pero mira los costos: entre 0,20 y 0,25 la
diferencia es de 4.732 contra 4.650 sin calibrar y 4.722 contra 4.658 con Platt, menos de 2%. La curva de
costo es plana cerca del óptimo, y con un modelo calibrado la fórmula cae dentro de esa zona plana aunque no
dé el mínimo exacto de la tabla. Esa es la utilidad concreta de calibrar: con probabilidades que significan lo
que dicen, el umbral se puede fijar con lápiz y papel a partir de los costos, sin depender de la muestra de
testing, y la misma probabilidad sirve para provisionar y para ponerle precio al riesgo. Y cuando la
descalibración es grande, como en el modelo balanceado de German Credit, la fórmula sin calibrar no cae en
ninguna zona plana: cae lejos.



## 7. Lo que hay que llevarse

1. Un clasificador hace dos cosas: ordenar y ponerle número al riesgo. El AUC mide lo primero; el diagrama de
   confiabilidad, el Brier, el log loss y el ECE miden lo segundo. Calibrar nunca mejora el orden.
2. La calibración se pierde por tres caminos: ponderar o remuestrear clases, una forma de modelo equivocada,
   y los árboles y ensambles por construcción. La logística entrenada sin pesos y bien especificada nace
   calibrada.
3. Si balanceaste una logística, resta $\ln(w_1/w_0)$ al logit: exacto, gratis y sin datos extra.
4. En el caso general, Platt (dos parámetros, corrige nivel y escala) o isotónica (corrige forma, necesita
   miles de filas), siempre ajustadas sobre datos que el modelo no vio y que no son testing.
5. Mira el diagrama antes de calibrar: un modelo ya calibrado no mejora y puede empeorar.
6. Con probabilidades calibradas, el umbral con costos se puede fijar con la fórmula, y la probabilidad sirve
   para provisionar y para poner precio al riesgo, no solo para ordenar.



## Ejercicios

1. **Submuestreo.** En el notebook 5 el submuestreo dejaba 210 buenos pagadores de los 490. Entrena esa
   logística (todos los malos más 210 buenos al azar, semilla 11), mide su calibración en testing, y aplica la
   corrección del intercepto con la constante $\ln(490/210)$. ¿Vuelve a la diagonal como con class weights?
2. **Calibrar sobre las mismas filas.** Repite Platt e isotónica para el random forest de hojas de 1, pero
   ajustando la función de calibración sobre la parte de **ajuste** (las mismas filas con que se entrenó el
   modelo) en vez de la de calibración. Compara el ECE en testing con el de la sección 5. ¿Qué pasó y por qué?
3. **Calibrar el boosting.** El boosting ya estaba calibrado. Aplica `CalibratedClassifierCV` con `cv=5` y
   `method="isotonic"` sobre training completo y compara AUC, Brier y ECE en testing con el boosting sin
   calibrar. ¿Cambió el AUC? ¿Por qué, si calibrar no cambia el orden?



## Soluciones


In [ ]:

# Ejercicio 1: submuestreo y corrección del intercepto
rng_b = np.random.default_rng(11)
idx_buenos = rng_b.choice(np.where(y_tr == 0)[0], size=n1, replace=False)
idx_sub = np.r_[np.where(y_tr == 1)[0], idx_buenos]
sub = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[idx_sub], y_tr[idx_sub])
z_sub = sub.decision_function(X_te)
p_sub = 1 / (1 + np.exp(-z_sub))
p_sub_corr = 1 / (1 + np.exp(-(z_sub - np.log(n0 / n1))))     # ln(490/210): los buenos quedaron subrepresentados en esa razón
print(tabla_metricas(y_te, {"submuestreo": p_sub, "submuestreo + corrección": p_sub_corr, "sin balancear": p_base}))
diagrama(y_te, {"submuestreo": p_sub, "submuestreo + corrección": p_sub_corr}, "Ejercicio 1: submuestreo antes y después de corregir el intercepto").show()



La corrección funciona igual: el submuestreo equivale a ponderar a los buenos por 210/490, así que la
constante es la misma $\ln(490/210)$, y la curva vuelve cerca de la diagonal. Queda un poco más ruidosa que
con class weights porque el modelo se entrenó con 420 filas en vez de 700: el submuestreo tira información,
y eso la corrección no lo recupera.


In [ ]:

# Ejercicio 2: calibrar sobre las filas de ajuste (lo que NO hay que hacer)
rf1 = modelos_tw["random forest, hojas de 1"]
p_fit = rf1.predict_proba(A_fit)[:, 1]
print("ECE del random forest sobre sus propias filas de ajuste:", round(ece(y_fit, p_fit), 3), " | promedio de p en positivos:", round(p_fit[y_fit == 1].mean(), 3), " en negativos:", round(p_fit[y_fit == 0].mean(), 3))
mal_platt = calibrar_platt(p_fit, y_fit, P_te["random forest, hojas de 1"])
mal_iso = calibrar_isotonica(p_fit, y_fit, P_te["random forest, hojas de 1"])
print(tabla_metricas(yt_te, {"sin calibrar": P_te["random forest, hojas de 1"], "Platt sobre calibración (bien)": P_platt["random forest, hojas de 1"], "Platt sobre ajuste (mal)": mal_platt,
                             "isotónica sobre calibración (bien)": P_iso["random forest, hojas de 1"], "isotónica sobre ajuste (mal)": mal_iso}))



Sobre sus propias filas de ajuste el random forest se ve mal calibrado, pero al revés que en testing: como
las memorizó, a los que no pagaron les pone probabilidades altas y a los que pagaron probabilidades bajas, y
la realidad ahí es exactamente 1 o 0. Es decir, visto desde sus propias filas el modelo parece **demasiado
tímido** (dice 0,8 y pasa 1; dice 0,1 y pasa 0). Una función de calibración aprendida sobre eso aprende que
"el modelo es más confiable de lo que dice" y empuja las probabilidades de testing hacia los extremos, donde
no corresponden: la probabilidad promedio cae de 0,22 a 0,15, y el ECE y el log loss en testing empeoran en
vez de mejorar (con la isotónica, el log loss se dispara). Es la misma regla de siempre con otro disfraz:
nada que se aprenda puede aprenderse sobre las filas que el modelo ya usó.


In [ ]:

# Ejercicio 3: CalibratedClassifierCV sobre el boosting (training completo, 5 pliegues)
t0 = time.time()
gb_cal = CalibratedClassifierCV(GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0), method="isotonic", cv=5).fit(A_tr, yt_tr)
p_gb_cal = gb_cal.predict_proba(A_te)[:, 1]
gb_solo = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_tr, yt_tr)
p_gb = gb_solo.predict_proba(A_te)[:, 1]
print(f"({time.time() - t0:.0f} s)")
print(tabla_metricas(yt_te, {"boosting, training completo": p_gb, "CalibratedClassifierCV isotónica, cv=5": p_gb_cal}))



El Brier y el ECE casi no se mueven, como corresponde a un modelo que ya estaba calibrado. Pero el AUC sí
cambia un poco, y eso parece contradecir la regla. La explicación es que `CalibratedClassifierCV` con `cv=5`
no calibra un modelo: entrena cinco boostings sobre cinco pliegues, calibra cada uno sobre el pliegue que
quedó fuera, y **promedia** los cinco. Ese promedio es un pequeño ensamble, y un ensamble sí puede cambiar el
orden. Cuando se quiera medir solo el efecto de calibrar, hay que hacerlo a mano como en la sección 5, con un
solo modelo y una partición de calibración separada.
